**Algorithme ID3** (Arbre de décision) 
L'algorithme ID3 construit un arbre de décision de manière récursive en utilisant deux concepts mathématiques clés :

1. **L'Entropie** : Mesure l'impureté ou le désordre dans un ensemble de données. $H(S) = - \sum p_i \log_2(p_i)$
2. **Le Gain d'Information** : Mesure la réduction de l'entropie après avoir séparé les données selon un attribut spécifique.


In [13]:
import math
from collections import Counter

# Calcul de l'entropie d'un ensemble de données
def calculer_entropie(donnees, index_cible):
    """
    Calcule l'entropie pour la colonne cible (la variable à prédire).
    """
    # Extraire toutes les valeurs de la colonne cible (ex: Oui/Non)
    valeurs_cibles = [ligne[index_cible] for ligne in donnees]
    
    # Compter l'occurrence de chaque classe (ex: 3 Oui, 2 Non)
    compteur_classes = Counter(valeurs_cibles)
    
    entropie = 0.0
    total_lignes = len(donnees)
    
    # Appliquer la formule de l'entropie : - Somme(p * log2(p))
    for nb_occurrences in compteur_classes.values():
        probabilite = nb_occurrences / total_lignes
        entropie -= probabilite * math.log2(probabilite)
        
    return entropie

# Séparation des données selon un attribut
def separer_donnees(donnees, index_attribut, valeur):
   
    sous_ensemble = []
    for ligne in donnees:
        if ligne[index_attribut] == valeur:
            sous_ensemble.append(ligne)
    return sous_ensemble

# Calcul du Gain d'Information
def gain_information(donnees, index_attribut, index_cible):

    # Entropie avant la division
    entropie_totale = calculer_entropie(donnees, index_cible)
    
    # Trouver toutes les valeurs uniques de l'attribut (ex: Soleil, Pluie, Nuages)
    valeurs_uniques = set([ligne[index_attribut] for ligne in donnees])
    
    entropie_apres_division = 0.0
    total_lignes = len(donnees)
    
    # Calculer l'entropie pondérée pour chaque sous-ensemble
    for valeur in valeurs_uniques:
        sous_ensemble = separer_donnees(donnees, index_attribut, valeur)
        probabilite = len(sous_ensemble) / total_lignes
        entropie_apres_division += probabilite * calculer_entropie(sous_ensemble, index_cible)
        
    # Le gain est la différence entre l'entropie totale et l'entropie après division
    return entropie_totale - entropie_apres_division

Maintenant que nous avons les outils mathématiques, nous pouvons écrire la fonction récursive qui va construire l'arbre de décision. L'arbre sera représenté sous forme d'un dictionnaire imbriqué.



In [14]:
def construire_arbre_id3(donnees, noms_attributs, index_cible):

    valeurs_cibles = [ligne[index_cible] for ligne in donnees]
    
    # Condition d'arrêt 1 : Si toutes les données ont la même cible (pur à 100%)
    if len(set(valeurs_cibles)) == 1:
        return valeurs_cibles[0] # Retourner la classe (ex: "Oui")
    
    # Condition d'arrêt 2 : S'il n'y a plus d'attributs à tester
    if len(noms_attributs) == 0:
        # Retourner la classe majoritaire
        return Counter(valeurs_cibles).most_common(1)[0][0]
    
    # Trouver le meilleur attribut pour diviser les données (celui avec le meilleur gain)
    gains = []
    for i in range(len(noms_attributs)):
        gain = gain_information(donnees, i, index_cible)
        gains.append(gain)
        
    index_meilleur_attribut = gains.index(max(gains))
    meilleur_attribut = noms_attributs[index_meilleur_attribut]
    
    # Initialiser le nœud de l'arbre avec le meilleur attribut
    arbre = {meilleur_attribut: {}}
    
    # Trouver les valeurs possibles pour ce meilleur attribut
    valeurs_possibles = set([ligne[index_meilleur_attribut] for ligne in donnees])
    
    # Mettre à jour la liste des attributs restants 
    attributs_restants = noms_attributs.copy()
    attributs_restants.pop(index_meilleur_attribut)
    
    # Créer les branches
    for valeur in valeurs_possibles:
        # Séparer les données
        sous_ensemble = separer_donnees(donnees, index_meilleur_attribut, valeur)
        
        # Enlever la colonne du meilleur attribut dans le sous-ensemble de données
        sous_ensemble_nettoye = []
        for ligne in sous_ensemble:
            nouvelle_ligne = ligne[:index_meilleur_attribut] + ligne[index_meilleur_attribut+1:]
            sous_ensemble_nettoye.append(nouvelle_ligne)
            
        # Appel récursif pour construire la suite de l'arbre
        sous_arbre = construire_arbre_id3(sous_ensemble_nettoye, attributs_restants, index_cible - 1 if index_cible > index_meilleur_attribut else index_cible)
        
        # Attacher le sous-arbre à la branche correspondante
        arbre[meilleur_attribut][valeur] = sous_arbre
        
    return arbre

Nous allons tester notre algorithme sur un jeu de données classique : "Doit-on jouer au tennis en fonction de la météo ?". Les données sont représentées sous forme de liste de listes 

In [15]:
import pprint # Pour afficher l'arbre de manière lisible

# Définition des colonnes
noms_colonnes = ["Meteo", "Temperature", "Humidite", "Vent"]

# Notre jeu de données d'entraînement (La dernière colonne est la cible "Jouer_Tennis")
donnees_entrainement = [
    ["Soleil", "Chaud", "Haute", "Faible", "Non"],
    ["Soleil", "Chaud", "Haute", "Fort", "Non"],
    ["Nuages", "Chaud", "Haute", "Faible", "Oui"],
    ["Pluie", "Doux", "Haute", "Faible", "Oui"],
    ["Pluie", "Frais", "Normale", "Faible", "Oui"],
    ["Pluie", "Frais", "Normale", "Fort", "Non"],
    ["Nuages", "Frais", "Normale", "Fort", "Oui"],
    ["Soleil", "Doux", "Haute", "Faible", "Non"],
    ["Soleil", "Frais", "Normale", "Faible", "Oui"],
    ["Pluie", "Doux", "Normale", "Faible", "Oui"],
    ["Soleil", "Doux", "Normale", "Fort", "Oui"],
    ["Nuages", "Doux", "Haute", "Fort", "Oui"],
    ["Nuages", "Chaud", "Normale", "Faible", "Oui"],
    ["Pluie", "Doux", "Haute", "Fort", "Non"]
]

# L'index de notre variable cible 
index_variable_cible = -1 

arbre_decision = construire_arbre_id3(donnees_entrainement, noms_colonnes, index_variable_cible)

print("\nArbre de décision généré :")
# pprint permet une jolie indentation des dictionnaires
pprint.pprint(arbre_decision)


Arbre de décision généré :
{'Meteo': {'Nuages': 'Oui',
           'Pluie': {'Vent': {'Faible': 'Oui', 'Fort': 'Non'}},
           'Soleil': {'Humidite': {'Haute': 'Non', 'Normale': 'Oui'}}}}


# prédire une nouvelle ligne de données

In [16]:
def faire_prediction(arbre, ligne_donnee, noms_attributs):  
    # Trouver l'attribut à la racine de l'arbre (le nœud actuel)
    noeud_actuel = list(arbre.keys())[0]
    
    # Trouver la valeur de cet attribut dans notre nouvelle donnée
    index_attribut = noms_attributs.index(noeud_actuel)
    valeur_donnee = ligne_donnee[index_attribut]
    
    # Parcourir la branche correspondante
    try:
        sous_arbre = arbre[noeud_actuel][valeur_donnee]
    except KeyError:
        # Si la valeur n'a jamais été vue lors de l'entraînement, on ne peut pas prédire (limite de ID3)
        return "Valeur inconnue"
    
    # Si le sous_arbre est un dictionnaire, on doit descendre plus bas (récursivité)
    if isinstance(sous_arbre, dict):
        return faire_prediction(sous_arbre, ligne_donnee, noms_attributs)
    else:
        # Sinon, on est arrivé à une feuille, on retourne le résultat !
        return sous_arbre

# Testons la prédiction sur un nouveau jour
nouveau_jour = ["Soleil", "Frais", "Haute", "Fort"]
resultat = faire_prediction(arbre_decision, nouveau_jour, ["Meteo", "Temperature", "Humidite", "Vent"])

print(f"\nDonnées du jour : {nouveau_jour}")
print(f"Prédiction (Doit-on jouer ?) : {resultat}")


Données du jour : ['Soleil', 'Frais', 'Haute', 'Fort']
Prédiction (Doit-on jouer ?) : Non
